In [1]:
from google import genai
from dotenv import load_dotenv
import os
load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL_NAME = "gemma-4-26b-a4b-it"

In [2]:
from langchain.tools import tool
from langchain_core.messages import ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
@tool
def add (a:int , b:int) -> int:
    """calculate the sum of these two numbers"""
    return a+b

In [4]:
model = ChatGoogleGenerativeAI(
    model=MODEL_NAME
)

In [5]:
model_with_tools = model.bind_tools([add])

In [6]:
user_message = {
    'role' : 'user',
    'content' : 'what is sum of 5 and 7'
}

In [7]:
response = model_with_tools.invoke([user_message])
response

AIMessage(content=[{'type': 'thinking', 'thinking': 'The user is asking for the sum of 5 and 7. I should use the `add` tool to calculate this.'}], additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"b": 7, "a": 5}'}, '__gemini_function_call_thought_signatures__': {'call_1864480': 'EiYKJGUyNDgzMGE3LTVjZDYtNDJmZS05OThiLWVlNTM5ZTcyYjljMw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemma-4-26b-a4b-it', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a042e2-3492-7751-8ef3-6d03f1c333b8-0', tool_calls=[{'name': 'add', 'args': {'b': 7, 'a': 5}, 'id': 'call_1864480', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 42, 'total_tokens': 106, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 26}})

In [8]:
tool_call = response.tool_calls[0]
tool_call

{'name': 'add',
 'args': {'b': 7, 'a': 5},
 'id': 'call_1864480',
 'type': 'tool_call'}

In [9]:
tool_result = add.invoke(tool_call['args'])
tool_result

12

In [10]:
tool_message = ToolMessage(
    content=str(tool_result),
    tool_call_id = tool_call['id']
)
tool_message

ToolMessage(content='12', tool_call_id='call_1864480')

In [11]:
messages = [user_message,tool_message,response]

In [ ]:
final_response = model_with_tools.invoke(messages)

In [15]:
final_response

AIMessage(content=[{'type': 'thinking', 'thinking': "The tool returned 12. I should now answer the user's question.\n"}, {'type': 'text', 'text': 'The sum of 5 and 7 is 12.'}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemma-4-26b-a4b-it', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a042e5-9e5a-7b52-a0ab-da202c8f9cbb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 31, 'total_tokens': 174, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 18}})

In [14]:
print("\nFINAL RESPONSE:")
print(final_response.content)


FINAL RESPONSE:
[{'type': 'thinking', 'thinking': "The tool returned 12. I should now answer the user's question.\n"}, {'type': 'text', 'text': 'The sum of 5 and 7 is 12.'}]
